In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(in_features=64 * 8 * 8, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool1(self.relu(self.conv1(x)))
        x = self.pool2(self.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
trainset = torchvision.datasets.SVHN(root='./data', split='train', download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=2)
testset = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False, num_workers=2)
classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')
model = SimpleCNN()
print("Model initialized successfully for SVHN!")
print(model)

100%|██████████| 182M/182M [00:11<00:00, 16.0MB/s]
100%|██████████| 64.3M/64.3M [00:05<00:00, 12.5MB/s]


Model initialized successfully for SVHN!
SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
epochs = 3
print("Starting Training...")
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if i % 500 == 499:
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 500:.3f}')
            running_loss = 0.0
print("Finished Training!")

Starting Training...
[Epoch 1, Batch 500] loss: 1.406
[Epoch 1, Batch 1000] loss: 0.662
[Epoch 1, Batch 1500] loss: 0.586
[Epoch 1, Batch 2000] loss: 0.545
[Epoch 2, Batch 500] loss: 0.457
[Epoch 2, Batch 1000] loss: 0.455
[Epoch 2, Batch 1500] loss: 0.431
[Epoch 2, Batch 2000] loss: 0.428
[Epoch 3, Batch 500] loss: 0.376
[Epoch 3, Batch 1000] loss: 0.375
[Epoch 3, Batch 1500] loss: 0.365
[Epoch 3, Batch 2000] loss: 0.370
Finished Training!


In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy of the network on the test images: {100 * correct // total} %')

Accuracy of the network on the test images: 86 %


In [ ]:
dataiter = iter(trainloader)
images, labels = next(dataiter)

print(f"{'Layer / Operation':<40} | {'Output Tensor Shape (Batch, C, H, W)'}")
print("-" * 75)

print(f"{'Input Image (Batch Size = 32)':<40} | {tuple(images.shape)}")

out = model.conv1(images)
print(f"{'Conv Layer 1 (32 filters, 3x3, P=Same)':<40} | {tuple(out.shape)}")

out = model.relu(out)
out = model.pool1(out)
print(f"{'Max Pool 1 (2x2, S=2)':<40} | {tuple(out.shape)}")

out = model.conv2(out)
print(f"{'Conv Layer 2 (64 filters, 3x3, P=Same)':<40} | {tuple(out.shape)}")

out = model.relu(out)
out = model.pool2(out)
print(f"{'Max Pool 2 (2x2, S=2)':<40} | {tuple(out.shape)}")

out = torch.flatten(out, 1)
print(f"{'Flatten Layer (1D Vector)':<40} | {tuple(out.shape)}")

out = model.fc1(out)
out = model.relu(out)
print(f"{'Fully Connected 1 (128 units)':<40} | {tuple(out.shape)}")

out = model.fc2(out)
print(f"{'Fully Connected 2 (10 units)':<40} | {tuple(out.shape)}")

Layer / Operation                        | Output Tensor Shape (Batch, C, H, W)
---------------------------------------------------------------------------
Input Image (Batch Size = 32)            | (32, 3, 32, 32)
Conv Layer 1 (32 filters, 3x3, P=Same)   | (32, 32, 32, 32)
Max Pool 1 (2x2, S=2)                    | (32, 32, 16, 16)
Conv Layer 2 (64 filters, 3x3, P=Same)   | (32, 64, 16, 16)
Max Pool 2 (2x2, S=2)                    | (32, 64, 8, 8)
Flatten Layer (1D Vector)                | (32, 4096)
Fully Connected 1 (128 units)            | (32, 128)
Fully Connected 2 (10 units)             | (32, 10)
